# From Detection to Trajectories: Mastering Multi-Object Tracking with Roboflow Trackers & OpenCV

## 🎓 What You'll Learn

- 🎯 Implement **ByteTrack** and **SORT** tracking algorithms
- 🎨 Visualize tracking with boxes, labels, IDs, and trajectories
- 🚁 Handle **moving cameras** with motion compensation
- 🧩 Track objects using **segmentation masks**
- 💻 Use both **CLI** and **Python API** approaches
- ⚡ Process and analyze real-world video footage

Let's dive in! 🚀

## ⚡ Checking GPU
As we are not only trackign but also detecting , we will be needing GPU

In [ ]:
!nvidia-smi

## 📂 Cloning the Repository

First, let's clone the GitHub repository which contains the sample videos
that we will be using for object detection and tracking throughout this notebook.

In [ ]:
# Clone only the specific folder from LearnOpenCV GitHub
!git clone --no-checkout https://github.com/Sudip-329/learnopencv.git
%cd learnopencv
!git sparse-checkout init --cone
!git sparse-checkout set Roboflow_Trackers_Demo
!git checkout

## 📦 Install dependencies

You may see dependency conflict warnings in Google Colab. This is expected for the preinstalled Google Colab environment and does not affect functionality.

In [ ]:
!pip install -q inference-gpu trackers==2.2.0

In [ ]:
!pip install pycuda

## 🐍 Object Tracking with ByteTrack - Python Approach

In this section, we will implement multi-object tracking using the **Python API** approach.
Here are the key steps we will follow:

- 🔍 **Detect** objects in each frame using **RF-DETR Medium**
- 🎯 **Track** detected objects across frames using **ByteTrack**
- 🎨 **Annotate** each frame with bounding boxes, track IDs and trajectories
- 🎬 **Export** the final tracked video

In [ ]:
from inference import get_model
from trackers import ByteTrackTracker

# Load RF-DETR medium model for object detection
model = get_model("rfdetr-medium")
# Initialize ByteTrack tracker
tracker = ByteTrackTracker()

In [ ]:
import supervision as sv

# Define a custom color palette for visualization
# Each tracked object will get a unique color based on its track ID
color = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
    "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

# Bounding box annotator — draws boxes around detected objects
box_annotator = sv.BoxAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK)

# Label annotator — displays track ID on each detected object
label_annotator = sv.LabelAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
    text_color=sv.Color.BLACK,
    text_scale=0.8)

# Trace annotator — draws trajectory trail for each tracked object
trace_annotator = sv.TraceAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
    thickness=2,
    trace_length=100)

In [ ]:
CONFIDENCE_THRESHOLD = 0.2
NMS_THRESHOLD = 0.3

# Input and output video paths
SOURCE_VIDEO_PATH_sample_1 = "/content/learnopencv/Roboflow_Trackers_Demo/Sample_1_Birds_Flying.mp4"
TARGET_VIDEO_PATH_sample_1 = "/content/Sample_1_Birds_Flying_output.mp4"

def callback(frame, i):
    result = model.infer(frame, confidence=CONFIDENCE_THRESHOLD)[0]
    detections = sv.Detections.from_inference(result).with_nms(threshold=NMS_THRESHOLD)
    detections = tracker.update(detections)

    annotated_image = frame.copy()
    annotated_image = box_annotator.annotate(annotated_image, detections)
    annotated_image = trace_annotator.annotate(annotated_image, detections)
    annotated_image = label_annotator.annotate(annotated_image, detections, detections.tracker_id)

    return annotated_image

# Reset tracker before processing
tracker.reset()

# Process the full video
sv.process_video(
    source_path=SOURCE_VIDEO_PATH_sample_1,
    target_path=TARGET_VIDEO_PATH_sample_1,
    callback=callback,
    show_progress=True,
)

In [ ]:
# Compress the output video for display in notebook
TARGET_VIDEO_sample_1_compressed = "/content/Sample_1_Birds_Flying_output_compressed.mp4"

!ffmpeg -y -loglevel error -i {TARGET_VIDEO_PATH_sample_1} -vcodec libx264 -crf 28 {TARGET_VIDEO_sample_1_compressed}

In [ ]:
# Display the output video in the notebook
from IPython.display import Video

Video(TARGET_VIDEO_sample_1_compressed, embed=True, width=1080)

### Let's Test Another Challenging Sample

In [ ]:
from inference import get_model
from trackers import ByteTrackTracker

# Load RF-DETR medium model for object detection
model = get_model("rfdetr-medium")
# Initialize ByteTrack tracker
tracker = ByteTrackTracker()

import supervision as sv

# Define a custom color palette for visualization
# Each tracked object will get a unique color based on its track ID
color = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
    "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

# Bounding box annotator — draws boxes around detected objects
box_annotator = sv.BoxAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK)

# Label annotator — displays track ID on each detected object
label_annotator = sv.LabelAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
    text_color=sv.Color.BLACK,
    text_scale=0.8)

# Trace annotator — draws trajectory trail for each tracked object
trace_annotator = sv.TraceAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
    thickness=2,
    trace_length=100)

CONFIDENCE_THRESHOLD = 0.2
NMS_THRESHOLD = 0.3

# Input and output video paths
SOURCE_VIDEO_PATH_sample_5 = "/content/learnopencv/Roboflow_Trackers_Demo/Sample_5_Snowboarding.mp4"
TARGET_VIDEO_PATH_sample_5 = "/content/Sample_5_Snowboarding.mp4"

def callback(frame, i):
    result = model.infer(frame, confidence=CONFIDENCE_THRESHOLD)[0]
    detections = sv.Detections.from_inference(result).with_nms(threshold=NMS_THRESHOLD)
    detections = tracker.update(detections)

    annotated_image = frame.copy()
    annotated_image = box_annotator.annotate(annotated_image, detections)
    annotated_image = trace_annotator.annotate(annotated_image, detections)
    annotated_image = label_annotator.annotate(annotated_image, detections, detections.tracker_id)

    return annotated_image

# Reset tracker before processing
tracker.reset()

# Process the full video
sv.process_video(
    source_path=SOURCE_VIDEO_PATH_sample_5,
    target_path=TARGET_VIDEO_PATH_sample_5,
    callback=callback,
    show_progress=True,
)

In [ ]:
# Compress the output video for display in notebook
TARGET_VIDEO_sample_5_compressed = "/content/Sample_5_Snowboarding_compressed.mp4"

!ffmpeg -y -loglevel error -i {TARGET_VIDEO_PATH_sample_5} -vcodec libx264 -crf 28 {TARGET_VIDEO_sample_5_compressed}

# Display the output video in the notebook
from IPython.display import Video

Video(TARGET_VIDEO_sample_5_compressed, embed=True, width=1080)

Hold on! ⚠️ That is something we are not expecting. Notice carefully that the trajectories are not correct and there must be something wrong. Yes, you are correct to catch it because here the camera is moving along with the objects, and that's causing these wrong trajectories.

Roboflow mentioned a way to solve this: **Camera Motion Compensation**

How it works? 👉 This code uses **MotionEstimator** to track camera movement by analyzing feature points between frames, then applies a coordinate transformation to compensate for the camera motion. The **MotionAwareTraceAnnotator** uses this transformation to draw stable trajectories that remain fixed relative to the ground, not the moving camera.

The below code implements this technique:

In [ ]:
from trackers import MotionEstimator, MotionAwareTraceAnnotator
PERSON_CLASS_ID = 0

model = get_model("rfdetr-large")
tracker = ByteTrackTracker(minimum_consecutive_frames=3)
motion_estimator = MotionEstimator(
    max_points=500,
    min_distance=10,
    quality_level=0.001,
    ransac_reproj_threshold=1.0,
)

color = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
    "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

box_annotator = sv.BoxAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK)

label_annotator = sv.LabelAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
    text_color=sv.Color.BLACK,
    text_scale=0.8)

motion_aware_trace_annotator = MotionAwareTraceAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
    thickness=2,
    trace_length=100)

In [ ]:
CONFIDENCE_THRESHOLD = 0.2
NMS_THRESHOLD = 0.3

# Input and output video paths
SOURCE_VIDEO_PATH_sample_5_stable = "/content/learnopencv/Roboflow_Trackers_Demo/Sample_5_Snowboarding.mp4"
TARGET_VIDEO_PATH_sample_5_stable = "/content/Sample_5_Snowboarding_stable.mp4"

def callback(frame, i):
    coord_transform = motion_estimator.update(frame)

    result = model.infer(frame, confidence=CONFIDENCE_THRESHOLD)[0]
    detections = sv.Detections.from_inference(result).with_nms(threshold=NMS_THRESHOLD)
    detections = detections[detections.class_id == PERSON_CLASS_ID]
    detections = tracker.update(detections)

    annotated_image = frame.copy()
    annotated_image = box_annotator.annotate(annotated_image, detections)
    annotated_image = motion_aware_trace_annotator.annotate(
        annotated_image, detections, coord_transform=coord_transform)
    annotated_image = label_annotator.annotate(annotated_image, detections, detections.tracker_id)

    return annotated_image

tracker.reset()
motion_estimator.reset()
motion_aware_trace_annotator.reset()

sv.process_video(
    source_path=SOURCE_VIDEO_PATH_sample_5_stable,
    target_path=TARGET_VIDEO_PATH_sample_5_stable,
    callback=callback,
    show_progress=True,
)

In [ ]:
TARGET_VIDEO_sample_5_compressed_stable =  "/content/Sample_5_Snowboarding_compressed_stable.mp4"

!ffmpeg -y -loglevel error -i {TARGET_VIDEO_PATH_sample_5_stable} -vcodec libx264 -crf 28 {TARGET_VIDEO_sample_5_compressed_stable}\

Video(TARGET_VIDEO_sample_5_compressed_stable, embed=True, width=1080)

**We will see more implementations afterwards...**

## 💻 Object Tracking with ByteTrack (CLI)

Roboflow Trackers also provides a simple **Command Line Interface (CLI)** for tracking,
which is a great option if you want to run tracking **without writing any Python code**!

Key highlights of the CLI approach:
- ⚡ **Faster to use** — just one command to run tracking
- 🛠️ **No coding required** — great for quick testing
- 🎯 **Same powerful models** — uses the same RF-DETR and ByteTrack under the hood
- 📹 **Trajectory visualization** — enabled with `--show-trajectories` flag

In [ ]:
SOURCE_VIDEO_PATH_sample_1_CLI = "/content/learnopencv/Roboflow_Trackers_Demo/Sample_1_Birds_Flying.mp4"
TARGET_VIDEO_PATH_sample_1_CLI = "/content/Sample_1_Birds_Flying_output_CLI.mp4"

# Run tracking using the CLI
# --source            : input video path
# --output            : output video path
# --model             : detection model to use
# --tracker           : tracking algorithm
# --show-trajectories : draws trajectory trails
!trackers track \
    --source {SOURCE_VIDEO_PATH_sample_1_CLI} \
    --output {TARGET_VIDEO_PATH_sample_1_CLI} \
    --model rfdetr-medium \
    --tracker bytetrack \
    --show-trajectories

In [ ]:
TARGET_VIDEO_sample_1_compressed_CLI =  "/content/Sample_1_Birds_Flying_output_CLI_compressed.mp4"

!ffmpeg -y -loglevel error -i {TARGET_VIDEO_PATH_sample_1_CLI} -vcodec libx264 -crf 28 {TARGET_VIDEO_sample_1_compressed_CLI}

In [ ]:
Video(TARGET_VIDEO_sample_1_compressed_CLI, embed=True, width=1080)

## 🎭 Object Tracking with Segmentation (Python API)

So far we have been using **bounding boxes** for tracking. Now let's take it a step further
with **instance segmentation** — where instead of just a box, the model will predict the
**exact mask/shape** of each detected object!

Key highlights:
- 🎯 **RF-DETR Segmentation Medium** — detects objects and predicts their masks
- 🎨 **Mask Annotator** — fills the detected object region with color
- 📦 **Bounding Box + Trajectory** — still shown alongside the mask
- 🔢 **Track IDs** — each object gets a unique ID across frames

In [ ]:
from inference import get_model
from trackers import ByteTrackTracker

# Load RF-DETR segmentation model for object detection + mask prediction
model = get_model("rfdetr-seg-medium")

# Initialize ByteTrack tracker
tracker = ByteTrackTracker()

In [ ]:
# Define a custom color palette — each track ID gets a unique color
color = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
    "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

# Bounding box annotator — draws boxes around detected objects
box_annotator = sv.BoxAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK)

# Mask annotator — fills the detected object region with color
mask_annotator = sv.MaskAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK)

# Label annotator — displays track ID on each detected object
label_annotator = sv.LabelAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
    text_color=sv.Color.BLACK,
    text_scale=0.8)

# Trace annotator — draws trajectory trail for each tracked object
trace_annotator = sv.TraceAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
    thickness=2,
    trace_length=100)

In [ ]:
CONFIDENCE_THRESHOLD = 0.2
NMS_THRESHOLD = 0.3

SOURCE_VIDEO_PATH_sample_1_seg = "/content/learnopencv/Roboflow_Trackers_Demo/Sample_1_Birds_Flying.mp4"
TARGET_VIDEO_PATH_sample_1_seg = "/content/Sample_1_Birds_Flying_output_seg.mp4"

def callback(frame, i):
    result = model.infer(frame, confidence=CONFIDENCE_THRESHOLD)[0]
    detections = sv.Detections.from_inference(result).with_nms(threshold=NMS_THRESHOLD)
    detections = tracker.update(detections)

    annotated_image = frame.copy()
    annotated_image = mask_annotator.annotate(annotated_image, detections)
    annotated_image = box_annotator.annotate(annotated_image, detections)
    annotated_image = trace_annotator.annotate(annotated_image, detections)
    annotated_image = label_annotator.annotate(annotated_image, detections, detections.tracker_id)

    return annotated_image

tracker.reset()

sv.process_video(
    source_path=SOURCE_VIDEO_PATH_sample_1_seg,
    target_path=TARGET_VIDEO_PATH_sample_1_seg,
    callback=callback,
    show_progress=True,
)

In [ ]:
TARGET_VIDEO_sample_1_compressed_seg = "/content/Sample_1_Birds_Flying_output_seg_compressed.mp4"

!ffmpeg -y -loglevel error -i {TARGET_VIDEO_PATH_sample_1_seg} -vcodec libx264 -crf 28 {TARGET_VIDEO_sample_1_compressed_seg}

In [ ]:
Video(TARGET_VIDEO_sample_1_compressed_seg, embed=True, width=1080)

# 🔄 Object Tracking using SORT

## 🐍 Track Using Python API

The process is exactly the same as ByteTrack — the only change is that we are now
using **SORTTracker** instead of **ByteTrackTracker**. SORT (Simple Online and
Realtime Tracking) is a lightweight and fast tracking algorithm that uses a
**Kalman Filter** for motion prediction and **IoU matching** for associating
detections across frames.

In [ ]:
from trackers import SORTTracker

# Load RF-DETR medium model for object detection
model = get_model("rfdetr-medium")

# Initialize SORT tracker
tracker = SORTTracker()

In [ ]:
import supervision as sv

color = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
    "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

box_annotator = sv.BoxAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK)

label_annotator = sv.LabelAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
    text_color=sv.Color.BLACK,
    text_scale=0.8)

trace_annotator = sv.TraceAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
    thickness=2,
    trace_length=100)

In [ ]:
CONFIDENCE_THRESHOLD = 0.2
NMS_THRESHOLD = 0.3

# Input and output video paths
SOURCE_VIDEO_PATH_sample_2 = "/content/learnopencv/Roboflow_Trackers_Demo/Sample_2_Sport.mp4"
TARGET_VIDEO_PATH_sample_2 = "/content/Sample_2_sport_output.mp4"

def callback(frame, i):
    result = model.infer(frame, confidence=CONFIDENCE_THRESHOLD)[0]
    detections = sv.Detections.from_inference(result).with_nms(threshold=NMS_THRESHOLD)
    detections = tracker.update(detections)

    annotated_image = frame.copy()
    annotated_image = box_annotator.annotate(annotated_image, detections)
    annotated_image = trace_annotator.annotate(annotated_image, detections)
    annotated_image = label_annotator.annotate(annotated_image, detections, detections.tracker_id)

    return annotated_image

# Reset tracker before processing
tracker.reset()

# Process the full video
sv.process_video(
    source_path=SOURCE_VIDEO_PATH_sample_2,
    target_path=TARGET_VIDEO_PATH_sample_2,
    callback=callback,
    show_progress=True,
)

In [ ]:
# Compress the output video for display in notebook
TARGET_VIDEO_sample_2_compressed = "/content/Sample_2_sport_output_compressed.mp4"

!ffmpeg -y -loglevel error -i {TARGET_VIDEO_PATH_sample_2} -vcodec libx264 -crf 28 {TARGET_VIDEO_sample_2_compressed}

In [ ]:
from IPython.display import Video

Video(TARGET_VIDEO_sample_2_compressed, embed=True, width=1080)

### Camera Motion Compensation



In [ ]:
from trackers import SORTTracker, MotionEstimator, MotionAwareTraceAnnotator

PERSON_CLASS_ID = 0

model = get_model("rfdetr-large")
tracker = SORTTracker(minimum_consecutive_frames=3)
motion_estimator = MotionEstimator(
    max_points=500,
    min_distance=10,
    quality_level=0.001,
    ransac_reproj_threshold=1.0,
)

color = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
    "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

box_annotator = sv.BoxAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK)

label_annotator = sv.LabelAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
    text_color=sv.Color.BLACK,
    text_scale=0.8)

motion_aware_trace_annotator = MotionAwareTraceAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
    thickness=2,
    trace_length=100)

In [ ]:
CONFIDENCE_THRESHOLD = 0.2
NMS_THRESHOLD = 0.3

# Input and output video paths
SOURCE_VIDEO_PATH_sample_2_stable = "/content/learnopencv/Roboflow_Trackers_Demo/Sample_2_Sport.mp4"
TARGET_VIDEO_PATH_sample_2_stable = "/content/Sample_2_sport_output_stable.mp4"

def callback(frame, i):
    coord_transform = motion_estimator.update(frame)

    result = model.infer(frame, confidence=CONFIDENCE_THRESHOLD)[0]
    detections = sv.Detections.from_inference(result).with_nms(threshold=NMS_THRESHOLD)
    detections = detections[detections.class_id == PERSON_CLASS_ID]
    detections = tracker.update(detections)

    annotated_image = frame.copy()
    annotated_image = box_annotator.annotate(annotated_image, detections)
    annotated_image = motion_aware_trace_annotator.annotate(
        annotated_image, detections, coord_transform=coord_transform)
    annotated_image = label_annotator.annotate(annotated_image, detections, detections.tracker_id)

    return annotated_image

tracker.reset()
motion_estimator.reset()
motion_aware_trace_annotator.reset()

sv.process_video(
    source_path=SOURCE_VIDEO_PATH_sample_2_stable,
    target_path=TARGET_VIDEO_PATH_sample_2_stable,
    callback=callback,
    show_progress=True,
)

In [ ]:
TARGET_VIDEO_sample_2_compressed_stable = "/content/Sample_2_sport_output_stable_compressed.mp4"

!ffmpeg -y -loglevel error -i {TARGET_VIDEO_PATH_sample_2_stable} -vcodec libx264 -crf 28 {TARGET_VIDEO_sample_2_compressed_stable}

In [ ]:
Video(TARGET_VIDEO_sample_2_compressed_stable, embed=True, width=1080)

## 💻 Object Tracking with SORT (CLI)

🎨 Customizing Your CLI Command

You can use the **CLI builder** on the Roboflow platform to configure:
- **Model size** — choose the detection model that fits your needs
- **Tracker** — select SORT or other available trackers
- **Visualization options** — control what appears in your output:
  - `--display` — show real-time output
  - `--boxes` — draw bounding boxes
  - `--masks` — show segmentation masks
  - `--confidence` — display confidence scores
  - `--labels` — show class labels
  - `--ids` — display tracking IDs
  - `--trajectories` — visualize object paths over time

Simply check the options you want in the CLI builder, and it will generate the complete command for you!

Visite Here: https://trackers.roboflow.com/latest/learn/track/#cli-command-builder

In [ ]:
SOURCE_VIDEO_PATH_sample_3 = "/content/learnopencv/Roboflow_Trackers_Demo/Sample_3_Run.mp4"
TARGET_VIDEO_PATH_sample_3 = "/content/sample_3_run_output.mp4"

!trackers track \
    --source {SOURCE_VIDEO_PATH_sample_3} \
    --output {TARGET_VIDEO_PATH_sample_3} \
    --model rfdetr-medium \
    --tracker sort \
    --show-trajectories

In [ ]:
TARGET_VIDEO_COMPRESSED_PATH_sample_3 = "/content/sample_3_run_output_compressed.mp4"

!ffmpeg -y -loglevel error -i {TARGET_VIDEO_PATH_sample_3} -vcodec libx264 -crf 28 {TARGET_VIDEO_COMPRESSED_PATH_sample_3}

In [ ]:
Video(TARGET_VIDEO_COMPRESSED_PATH_sample_3, embed=True, width=1080)

### Camera Motion Compensation

In [ ]:
PERSON_CLASS_ID = 0

model = get_model("rfdetr-large")
tracker = SORTTracker(minimum_consecutive_frames=3)
motion_estimator = MotionEstimator(
    max_points=500,
    min_distance=10,
    quality_level=0.001,
    ransac_reproj_threshold=1.0,
)

color = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
    "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

box_annotator = sv.BoxAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK)

label_annotator = sv.LabelAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
    text_color=sv.Color.BLACK,
    text_scale=0.8)

motion_aware_trace_annotator = MotionAwareTraceAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
    thickness=2,
    trace_length=100)

In [ ]:
CONFIDENCE_THRESHOLD = 0.2
NMS_THRESHOLD = 0.3

# Input and output video paths
SOURCE_VIDEO_PATH_sample_3_stable = "/content/learnopencv/Roboflow_Trackers_Demo/Sample_3_Run.mp4"
TARGET_VIDEO_PATH_sample_3_stable = "/content/sample_3_run_output_stable.mp4"

def callback(frame, i):
    coord_transform = motion_estimator.update(frame)

    result = model.infer(frame, confidence=CONFIDENCE_THRESHOLD)[0]
    detections = sv.Detections.from_inference(result).with_nms(threshold=NMS_THRESHOLD)
    detections = detections[detections.class_id == PERSON_CLASS_ID]
    detections = tracker.update(detections)

    annotated_image = frame.copy()
    annotated_image = box_annotator.annotate(annotated_image, detections)
    annotated_image = motion_aware_trace_annotator.annotate(
        annotated_image, detections, coord_transform=coord_transform)
    annotated_image = label_annotator.annotate(annotated_image, detections, detections.tracker_id)

    return annotated_image

tracker.reset()
motion_estimator.reset()
motion_aware_trace_annotator.reset()

sv.process_video(
    source_path=SOURCE_VIDEO_PATH_sample_3_stable,
    target_path=TARGET_VIDEO_PATH_sample_3_stable,
    callback=callback,
    show_progress=True,
)

In [ ]:
TARGET_VIDEO_COMPRESSED_PATH_sample_3_stable = "/content/sample_3_run_output_compressed_stable.mp4"

!ffmpeg -y -loglevel error -i {TARGET_VIDEO_PATH_sample_3_stable} -vcodec libx264 -crf 28 {TARGET_VIDEO_COMPRESSED_PATH_sample_3_stable}

Video(TARGET_VIDEO_COMPRESSED_PATH_sample_3_stable, embed=True, width=1080)

## 🧩 Object Tracking with Segmentation (Python API)

In [ ]:
model = get_model("rfdetr-seg-medium")
tracker = SORTTracker()

In [ ]:
color = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
    "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

box_annotator = sv.BoxAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK)

mask_annotator = sv.MaskAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK)

label_annotator = sv.LabelAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
    text_color=sv.Color.BLACK,
    text_scale=0.8)

trace_annotator = sv.TraceAnnotator(
    color=color,
    color_lookup=sv.ColorLookup.TRACK,
    thickness=2,
    trace_length=100)

In [ ]:
CONFIDENCE_THRESHOLD = 0.2
NMS_THRESHOLD = 0.3

SOURCE_VIDEO_PATH_sample_4 = "/content/learnopencv/Roboflow_Trackers_Demo/Sample_4_Birds.mp4"
TARGET_VIDEO_PATH_sample_4 = "/content/Sample_4_Birds_output.mp4"

def callback(frame, i):
    result = model.infer(frame, confidence=CONFIDENCE_THRESHOLD)[0]
    detections = sv.Detections.from_inference(result).with_nms(threshold=NMS_THRESHOLD)
    detections = tracker.update(detections)

    annotated_image = frame.copy()
    annotated_image = mask_annotator.annotate(annotated_image, detections)
    annotated_image = box_annotator.annotate(annotated_image, detections)
    annotated_image = trace_annotator.annotate(annotated_image, detections)
    annotated_image = label_annotator.annotate(annotated_image, detections, detections.class_id)

    return annotated_image

tracker.reset()

sv.process_video(
    source_path=SOURCE_VIDEO_PATH_sample_4,
    target_path=TARGET_VIDEO_PATH_sample_4,
    callback=callback,
    show_progress=True,
)

In [ ]:
TARGET_VIDEO_COMPRESSED_PATH_seg_sample_4 = "/content/Sample_4_Birds_output_seg_compressed.mp4"

!ffmpeg -y -loglevel error -i {TARGET_VIDEO_PATH_sample_4} -vcodec libx264 -crf 28 {TARGET_VIDEO_COMPRESSED_PATH_seg_sample_4}

Video(TARGET_VIDEO_COMPRESSED_PATH_seg_sample_4, embed=True, width=1080)

**🎊 Awesome! You've completed all the tracking implementations. Want more practice? Test it out on `sample_6` or challenge yourself with your own custom videos!**